###imports

In [1]:
!pip install lets-plot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 2.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

In [3]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from lets_plot import *
from lets_plot.mapping import as_discrete

In [4]:
LetsPlot.setup_html()

In [5]:
df_department = pd.read_csv("/content/department-v2.csv")
df_seniority = pd.read_csv("/content/seniority-v2.csv")

###data


In [6]:
def workers_positions_to_csv(data):
    people = json.loads(Path(data).read_text(encoding="utf-8"))
    rows = []

    for person_idx, jobs in enumerate(people, start=1):
        jobs_sorted = sorted(jobs, key=lambda d: (d.get("startDate") is None, d.get("startDate")))
        for job_idx, j in enumerate(jobs_sorted, start=1):
            rows.append({
                "person_id": person_idx,
                "job_no": job_idx,
                "organization": j.get("organization"),
                "linkedin": j.get("linkedin"),
                "position": j.get("position"),
                "startDate": j.get("startDate"),
                "endDate": j.get("endDate"),
                "status": j.get("status"),
                "department": j.get("department"),
                "seniority": j.get("seniority"),
            })

    df = pd.DataFrame(rows, columns=[
        "person_id","job_no","organization","linkedin","position",
        "startDate","endDate","status","department","seniority"
    ])
    df["person_id"] = df["person_id"].astype(int)
    df["job_no"] = df["job_no"].astype(int)
    return df

In [7]:
df_t_annotated= workers_positions_to_csv("/content/linkedin-cvs-annotated.json")
df_t_notannotated= workers_positions_to_csv("/content/linkedin-cvs-not-annotated.json")

In [8]:
df_t_annotated

,person_id,job_no,organization,linkedin,position,startDate,endDate,status,department,seniority
0,1,1,Nagel Car Group,,Buchhalterin,2000-05,2019-06,INACTIVE,Other,Professional
1,1,2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management
2,1,3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,None,ACTIVE,Other,Professional
3,1,4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,None,ACTIVE,Other,Management
4,1,5,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management
...,...,...,...,...,...,...,...,...,...,...
2633,608,2,Malteser Hilfsdienst,https://www.linkedin.com/company/malteser-hilf...,Präsidium,1999,None,ACTIVE,Other,Management
2634,608,3,FORESTA Management,,Geschäftsführer,2010-01,None,ACTIVE,Other,Management
2635,608,4,Bistum,,Justitiar,2014-03,None,ACTIVE,Other,Professional
2636,608,5,Malteserorden,,Kanzler der deutschen Assoziation,2015-01,None,ACTIVE,Other,Management


In [9]:
df_t_notannotated

,person_id,job_no,organization,linkedin,position,startDate,endDate,status,department,seniority
0,1,1,Domino's Pizza,,delivery/sales/MIT,2012-04,2012-06,INACTIVE,None,None
1,1,2,UPS Freight,https://www.linkedin.com/company/ups-freight,Regional driver,2016-10,2017-03,INACTIVE,None,None
2,1,3,Sysco,https://www.linkedin.com/company/sysco,Food Delivery Driver,2017-10,2018-02,INACTIVE,None,None
3,1,4,MB Railways,,Fuel Driver,2018-03,2019-03,INACTIVE,None,None
4,1,5,Playful Paws,,Co-Owner,2018-11,None,ACTIVE,None,None
...,...,...,...,...,...,...,...,...,...,...
1881,389,1,Euler Hermes Deutschland,https://www.linkedin.com/company/allianz-trade...,Juristischer Berater,1996-03,None,ACTIVE,None,None
1882,390,1,Folkwang Universität der Künste,,Dekanat,1998-02,2004-02,INACTIVE,None,None
1883,390,2,Folkwang Universität der Künste,,"Leitung Gebäudemanagement, Einkauf und Control...",2004-03,2005-09,INACTIVE,None,None
1884,390,3,Landesspracheninstitut in der Ruhr-Universität...,,Verwaltungsleitung Landesspracheninstitut in d...,2005-10,2015-06,INACTIVE,None,None


## Rule-based approach


###Apply rule-based approach on seniority level


####To predict the seniority level from job titles, we apply rule-based approach based on predefined job-title phrases. Firstly, all job titles and rule phrases are normalized to avoid special characters affecting a reliable match.
####Problem occured when an observation, for example, "senior sales marketing manager" is classified as "management", instead of "senior". This is because only "marketing manager" was picked first due to having smaller index compared to "senior sales marketing manager". To avoid this, we sort the phrases based on length, so that longer phrases are checked first because they are more specific.

In [10]:
import unicodedata
import re

# To normalize text by Lowercase, remove accents, keep letters/numbers, collapse spaces
def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^a-z0-9]+", " ", s)     # keep letters/numbers only
    s = re.sub(r"\s+", " ", s).strip()
    return s


df_seniority = df_seniority.copy()
df_seniority["text_norm"] = df_seniority["text"].map(normalize_text)
df_seniority["len"] = df_seniority["text_norm"].str.len()

df_seniority_sorted = df_seniority.sort_values(["len"], ascending=[False]).reset_index(drop=True) # Longest phrase first
compiled_rules_seniority = []

# Here we want to match whole phrase as a token sequence
for phrase, label in zip(df_seniority_sorted["text_norm"], df_seniority_sorted["label"]):
    pat = r"(?:^|\s)" + re.escape(phrase) + r"(?:\s|$)"  # to avoid partial matches
    compiled_rules_seniority.append((re.compile(pat), label, phrase))

####When there is not any predefined job-title phrase to match with, "professionall" will also be assigned, since it's the most occured seniority.

In [32]:

df_t_annotated_rb = df_t_annotated.copy()
df_t_annotated_rb["position_norm"] = df_t_annotated_rb["position"].map(normalize_text)

def predict_rule_based(pos_norm: str, compiled_rules):
    for rx, label, rule_phrase in compiled_rules:
        if rx.search(pos_norm):
            return (label, rule_phrase)   # first hit = best hit (because we sorted rules)
    return (None, None)

results = []

for pos in df_t_annotated_rb["position_norm"]:
    result = predict_rule_based(pos, compiled_rules_seniority)
    results.append(result)

pred_sen = pd.Series(results)

df_t_annotated_rb["pred_seniority"] = pred_sen.map(lambda x: x[0])
df_t_annotated_rb["matched_rule_seniority"]  = pred_sen.map(lambda x: x[1])

df_t_annotated_rb["pred_seniority_comb"] = (df_t_annotated_rb["pred_seniority"].fillna("Professional"))

#df_t_annotated_rb["pred_seniority_comb"] = df_t_annotated_rb["pred_seniority"]

#df_t_annotated_rb["pred_seniority_comb"] = (df_t_annotated_rb["pred_seniority"].fillna("UNMATCHED"))

#df_t_annotated_rb[["position", "seniority", "pred_seniority", "matched_rule_seniority","pred_seniority_comb"]].head(20)



####We then check for accuracy.

In [33]:
coverage = df_t_annotated_rb["pred_seniority"].notna().mean()
print("Coverage:", round(coverage * 100, 2), "%")

mask = df_t_annotated_rb["pred_seniority_comb"].notna() & df_t_annotated_rb["seniority"].notna()
acc = (df_t_annotated_rb.loc[mask, "pred_seniority_comb"] == df_t_annotated_rb.loc[mask, "seniority"]).mean()
print("Accuracy when replace none with Professional:", round(acc * 100, 2), "%")

acc = (df_t_annotated_rb["pred_seniority"] == df_t_annotated_rb["seniority"]).mean()
print("Accuracy using predefined phrases only:", round(acc * 100, 2), "%")

print("\n")


df_t_annotated_rb_without_Professional = (df_t_annotated_rb[df_t_annotated_rb["seniority"] != "Professional"].copy())
coverage = df_t_annotated_rb_without_Professional["pred_seniority"].notna().mean()
print("Coverage when ignore Professional labels:", round(coverage * 100, 2), "%")

acc = (df_t_annotated_rb_without_Professional["pred_seniority"] == df_t_annotated_rb_without_Professional["seniority"]).mean()
print("Accuracy overall when ignore Professional:", round(acc * 100, 2), "%")

print("\n")

print(pd.crosstab(df_t_annotated_rb.loc[mask, "seniority"],
                  df_t_annotated_rb.loc[mask, "pred_seniority_comb"]))

from sklearn.metrics import classification_report

mask = (
    df_t_annotated_rb["pred_seniority_comb"].notna() &
    df_t_annotated_rb["seniority"].notna()
)

y_true = df_t_annotated_rb.loc[mask, "seniority"]
y_pred = df_t_annotated_rb.loc[mask, "pred_seniority_comb"]

print(classification_report(y_true, y_pred, digits=4, zero_division=0))

#Top positions name that weren't detected
#misses = df_t_annotated_rb[df_t_annotated_rb["matched_rule_seniority"].isna()]
#misses["position"].value_counts().head(30)


Coverage: 52.81 %
Accuracy when replace none with Professional: 54.51 %
Accuracy using predefined phrases only: 25.74 %


Coverage when ignore Professional labels: 65.75 %
Accuracy overall when ignore Professional: 47.85 %


pred_seniority_comb  Director  Junior  Lead  Management  Professional  Senior
seniority                                                                    
Director                  117       0     0           0            18       6
Junior                      0      31     0           0           144      55
Lead                        1       5   209           5           164      76
Management                 41       1     9         208           118      41
Professional                0      56     0           3           759     401
Senior                      2       9     1           2            42     114
              precision    recall  f1-score   support

    Director     0.7267    0.8298    0.7748       141
      Junior     0.3039    0.1348    0.186

###Apply rule-based approach on department

####Similiar to predicting seniority, we apply the same logic to department matching.

In [34]:
df_department = df_department.copy()
df_department["text_norm"] = df_department["text"].map(normalize_text)
df_department["len"] = df_department["text_norm"].str.len()

df_department_sorted = df_department.sort_values(["len"], ascending=[False]).reset_index(drop=True) # Longest phrase first
compiled_rules_department = []

for phrase, label in zip(df_department_sorted["text_norm"], df_department_sorted["label"]):
    pat = r"(?:^|\s)" + re.escape(phrase) + r"(?:\s|$)"  # to avoid partial matches
    compiled_rules_department.append((re.compile(pat), label, phrase))


results = []

for pos in df_t_annotated_rb["position_norm"]:
    result = predict_rule_based(pos, compiled_rules_department)
    results.append(result)

pred_dep = pd.Series(results)

df_t_annotated_rb["pred_department"] = pred_dep.map(lambda x: x[0])
df_t_annotated_rb["matched_rule_department"]  = pred_dep.map(lambda x: x[1])

df_t_annotated_rb["pred_department_comb"] = (df_t_annotated_rb["pred_department"].fillna("Other"))

#df_t_annotated_rb[["position", "department", "pred_department", "matched_rule_department","pred_department_comb"]].head(20)

In [35]:
coverage = df_t_annotated_rb["pred_department"].notna().mean()
print("Coverage:", round(coverage * 100, 2), "%")

mask = df_t_annotated_rb["pred_department_comb"].notna() & df_t_annotated_rb["department"].notna()
acc = (df_t_annotated_rb.loc[mask, "pred_department_comb"] == df_t_annotated_rb.loc[mask, "department"]).mean()
print("Accuracy when replace None with Other:", round(acc * 100, 2), "%")

acc = (df_t_annotated_rb["pred_department"] == df_t_annotated_rb["department"]).mean()
print("Accuracy using predefined phrases only:", round(acc * 100, 2), "%")

#print(pd.crosstab(df_t_annotated_rb.loc[mask, "department"],df_t_annotated_rb.loc[mask, "pred_department"]))

#Top positions name that weren't detected
#misses = df_t_annotated_rb[df_t_annotated_rb["matched_rule_department"].isna()]
#misses["position"].value_counts().head(30)

mask = df_t_annotated_rb["department"].notna()

y_true = df_t_annotated_rb.loc[mask, "department"]
y_pred = df_t_annotated_rb.loc[mask, "pred_department_comb"]  # already fillna("Other")

labels = sorted(list(set(y_true.unique()).union(set(y_pred.unique()))))

print(classification_report(y_true, y_pred, labels=labels, digits=4, zero_division=0))

Coverage: 23.58 %
Accuracy when replace None with Other: 65.43 %
Accuracy using predefined phrases only: 19.41 %
                        precision    recall  f1-score   support

        Administrative     0.5333    0.0952    0.1616        84
  Business Development     0.5312    0.4359    0.4789        78
            Consulting     0.9429    0.5077    0.6600       195
      Customer Support     0.5000    0.0417    0.0769        48
       Human Resources     1.0000    0.1429    0.2500        70
Information Technology     0.7059    0.1923    0.3023       312
             Marketing     0.7206    0.3684    0.4876       133
                 Other     0.6025    0.9720    0.7439      1252
    Project Management     0.8878    0.4971    0.6374       175
            Purchasing     0.9583    0.3194    0.4792        72
                 Sales     0.9448    0.6256    0.7527       219

              accuracy                         0.6543      2638
             macro avg     0.7570    0.3817    0.4573

###Findings

####The rule-based approach based on predefined phrases demonstrates challenge to incorporate ambiguity in its prediction. Generic and overlapping job titles hinder performance, especially for department prediction.
####Seniority prediction based on predefined job-title could be applied to slightly more than half of the observation, with 52.81% on coverage. However, when restricting to only positions matched by predefined rules (excluding "Professional"), accuracy plummets to 25.74%. The approach performs well for clearly defined roles such as Director and Lead, but struggles with overlapping titles within Professinal, Senior, and Junior.
####Regarding department prediction, only 23.58% of the observation was covered. However, this method can predict  pretty accurately once it detects predefined phrases, as strict accuracy is at 19.41%, which is very close to coverage rate. Overall accuracy 65.43% is due to "Other" being the most occured catergory, which predefined phrases do not capture.

##Programmatic labeling + supervised learning

---





###Using rule-based approach to label

In [15]:
#Create a df for training using rule-based approach to fill pseudo labels

df_t_notannotated_for_training = df_t_notannotated.copy()
df_t_notannotated_for_training["position_norm"] = df_t_notannotated_for_training["position"].map(normalize_text)

#### Apply rule-based matching to seniority

In [16]:
df_t_notannotated_for_training["seniority"] = df_t_notannotated_for_training["position_norm"].apply(lambda x: predict_rule_based(x, compiled_rules_seniority)[0])


coverage = df_t_notannotated_for_training["seniority"].notna().mean()
print("Coverage seniority:", round(coverage * 100, 2), "%")

#df_t_notannotated_for_training[["position", "seniority"]].sample(20, random_state=42)

Coverage seniority: 58.11 %


####Apply rule-based matching to department

In [17]:
df_t_notannotated_for_training["department"] = df_t_notannotated_for_training["position_norm"].apply(lambda x: predict_rule_based(x, compiled_rules_department)[0])

coverage = df_t_notannotated_for_training["department"].notna().mean()
print("Coverage department:", round(coverage * 100, 2), "%")

#df_t_notannotated_for_training[["position", "department"]].sample(20, random_state=42)

Coverage department: 23.12 %


####Filter out empty rows and create new dataset for training purpose

In [18]:
coverage = (df_t_notannotated_for_training["department"].notna() & df_t_notannotated_for_training["seniority"].notna()).mean() * 100

print("Coverage (department & seniority):", round(coverage, 2), "%")

df_training_rb = df_t_notannotated_for_training[
    (df_t_notannotated_for_training["department"].notna()) |
    (df_t_notannotated_for_training["seniority"].notna())
].copy()

print("No. rows before:", len(df_t_notannotated_for_training))
print("No. rows after:", len(df_training_rb))

#output_path = "df_training_rb_included_na.csv"
#df_training_rb.to_csv(output_path, index=False)

Coverage (department & seniority): 17.92 %
No. rows before: 1886
No. rows after: 1194


###Apply Logistics Regression model for seniorty and department

#####Seniority

In [37]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Try different dataset
#df_exp = pd.read_csv('/content/df_t_notannotated_pred_embeddings_no_threshold.csv')
#df_exp = pd.read_csv('/content/df_t_notannotated_pred_embeddings_both_threshold.csv')
df_exp = pd.read_csv('/content/df_t_notannotated_pred_bow_no_thresholdv.csv')
#df_exp = pd.read_csv('/content/df_t_notannotated_pred_bow_both_threshold.csv')
df_exp["position_norm"] = df_exp["position"].map(normalize_text)

df_train_sen = df_exp.dropna(subset=["seniority"]).copy()
#df_train_sen = df_t_notannotated_for_training.dropna(subset=["seniority"]).copy()

X_sen = df_train_sen["position_norm"]
y_sen = df_train_sen["seniority"]

X_train, X_test, y_train, y_test = train_test_split(X_sen, y_sen, test_size=0.2, random_state=42, stratify=y_sen)

# CountVectorizer
count_vect = CountVectorizer(ngram_range=(1,1))
X_train_counts = count_vect.fit_transform(X_train)
X_test_counts  = count_vect.transform(X_test)
# Apply Tfid
tfidf_transformer = TfidfTransformer(use_idf=True)

X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)
X_test_tfidf  = tfidf_transformer.transform(X_test_counts)

# Running Logistics Regression model
clf = LogisticRegression().fit(X_train_tfidf, y_train)
y_pred = clf.predict(X_test_tfidf)

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))


Accuracy: 83.07 %
              precision    recall  f1-score   support

    Director       1.00      0.83      0.91        29
      Junior       0.90      0.47      0.62        19
        Lead       0.70      0.88      0.78        86
  Management       0.82      0.70      0.76        98
      Senior       0.90      0.93      0.92       146

    accuracy                           0.83       378
   macro avg       0.86      0.76      0.80       378
weighted avg       0.84      0.83      0.83       378



In [38]:
df_t_annotated_rb_test = df_t_annotated_rb.copy()
test_df = df_t_annotated_rb_test[df_t_annotated_rb_test["seniority"].notna() & (df_t_annotated_rb_test["seniority"] != "Professional")].copy()
X_test = test_df["position_norm"]
y_test = test_df["seniority"]

X_test_counts = count_vect.transform(X_test)
X_test_tfidf = tfidf_transformer.transform(X_test_counts)

y_pred = clf.predict(X_test_tfidf)
print("Accuracy on annotated set (without Professional):", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))

Accuracy on annotated set (without Professional): 63.5 %
              precision    recall  f1-score   support

    Director       0.73      0.80      0.76       141
      Junior       0.69      0.13      0.21       230
        Lead       0.69      0.73      0.71       460
  Management       0.82      0.67      0.74       418
      Senior       0.36      0.84      0.51       170

    accuracy                           0.63      1419
   macro avg       0.66      0.63      0.59      1419
weighted avg       0.69      0.63      0.62      1419



Overall LR has slightly better performance than rule-based approach (49.61% compared to 47.85%). The model struggles to identify Junior and Senior, and possibly mistakes between the two.
Notably, the performance based on dataset generated with pseudo-labels have very high accuracy of up to 93.645. This discrepancy could be the bias resulted from rule-based approach, suggesting poor generalization of the predefined phrases.


Now we incorporate "Professinonal" labelling using confidence level, assuming positions fall in this category if the highest confidence level of a position is lesser than 0.5.

In [39]:
test_df = df_t_annotated_rb.copy()
X_test = test_df["position_norm"]
y_test = test_df["seniority"]

X_test_counts = count_vect.transform(X_test)
X_test_tfidf = tfidf_transformer.transform(X_test_counts)

y_proba = clf.predict_proba(X_test_tfidf)
y_pred = clf.classes_[np.argmax(y_proba, axis=1)]
y_pred_confidence_lv = np.max(y_proba, axis=1)

test_df["y_pred"] = y_pred
test_df["y_pred_confidence_lv"] = y_pred_confidence_lv

In [40]:
threshold = 0.0
test_df["y_pred_after_confidence_lv"] = np.where(
    test_df["y_pred_confidence_lv"] < threshold,
    "Professional",
    test_df["y_pred"]
)

print("Accuracy with confidence threshold:", round(accuracy_score(test_df["seniority"], test_df["y_pred_after_confidence_lv"]) * 100, 2),"%")

print(classification_report(test_df["seniority"], test_df["y_pred_after_confidence_lv"]))

Accuracy with confidence threshold: 34.15 %
              precision    recall  f1-score   support

    Director       0.73      0.80      0.76       141
      Junior       0.31      0.13      0.18       230
        Lead       0.37      0.73      0.49       460
  Management       0.54      0.67      0.60       418
Professional       0.00      0.00      0.00      1219
      Senior       0.15      0.84      0.25       170

    accuracy                           0.34      2638
   macro avg       0.35      0.53      0.38      2638
weighted avg       0.22      0.34      0.25      2638



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


The performance of this approach seems worse than simple rule-based matching with "Professional" replacing non-detected position (46.89% < 54.51%). However, Logistics Regression offers more flexibility, and is better at predicting non-Professional titles, compared to rule-based approach (accuracy 49.61% > 47.85% when "Professional" is ruled out).

#####Department

In [41]:
#df_exp = pd.read_csv('/content/df_t_notannotated_pred_embeddings_no_threshold.csv')
#df_exp = pd.read_csv('/content/df_t_notannotated_pred_embeddings_both_threshold.csv')
#df_exp = pd.read_csv('/content/df_t_notannotated_pred_bow_no_thresholdv.csv')
#df_exp = pd.read_csv('/content/df_t_notannotated_pred_bow_both_threshold.csv')
df_exp = pd.read_csv('/content/df_notannotated_pred_embedding_bow.csv')

df_exp["position_norm"] = df_exp["position"].map(normalize_text)

df_train_dep = df_exp.dropna(subset=["department"]).copy()

#df_train_dep = df_t_notannotated_for_training.dropna(subset=["department"]).copy()

X_dep = df_train_dep["position_norm"]
y_dep = df_train_dep["department"]

X_train, X_test, y_train, y_test = train_test_split(X_dep, y_dep, test_size=0.2, random_state=42, stratify=y_dep)

# CountVectorizer
count_vect = CountVectorizer(ngram_range=(1,1))
X_train_counts = count_vect.fit_transform(X_train)
X_test_counts  = count_vect.transform(X_test)
# Apply Tfid
tfidf_transformer = TfidfTransformer(use_idf=True)

X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)
X_test_tfidf  = tfidf_transformer.transform(X_test_counts)

# Running Logistics Regression model
clf = LogisticRegression().fit(X_train_tfidf, y_train)
y_pred = clf.predict(X_test_tfidf)

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))

Accuracy: 79.1 %
                        precision    recall  f1-score   support

        Administrative       1.00      0.36      0.53        14
  Business Development       0.73      0.53      0.62        15
            Consulting       0.91      0.54      0.68        37
      Customer Support       1.00      0.33      0.50         3
       Human Resources       1.00      0.62      0.77         8
Information Technology       0.67      0.13      0.22        15
             Marketing       1.00      0.79      0.88        14
                 Other       0.75      1.00      0.86       219
    Project Management       0.92      0.63      0.75        19
            Purchasing       1.00      0.14      0.25         7
                 Sales       0.94      0.59      0.73        27

              accuracy                           0.79       378
             macro avg       0.90      0.52      0.62       378
          weighted avg       0.82      0.79      0.76       378



In [42]:
test_df = df_t_annotated_rb.copy()
X_test = test_df["position_norm"]
y_test = test_df["department"]

X_test_counts = count_vect.transform(X_test)
X_test_tfidf = tfidf_transformer.transform(X_test_counts)

y_pred = clf.predict(X_test_tfidf)
print("Accuracy on annotated set:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))

Accuracy on annotated set: 60.69 %
                        precision    recall  f1-score   support

        Administrative       0.18      0.08      0.11        84
  Business Development       0.36      0.41      0.39        78
            Consulting       0.81      0.53      0.64       195
      Customer Support       1.00      0.08      0.15        48
       Human Resources       0.93      0.37      0.53        70
Information Technology       0.91      0.10      0.18       312
             Marketing       0.68      0.26      0.37       133
                 Other       0.56      0.93      0.70      1252
    Project Management       0.98      0.45      0.62       175
            Purchasing       0.00      0.00      0.00        72
                 Sales       0.95      0.55      0.70       219

              accuracy                           0.61      2638
             macro avg       0.67      0.34      0.40      2638
          weighted avg       0.67      0.61      0.55      2638



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


There is some improvement compared to the rule-based approach. The model still cannot generalize real-data. This could be the result of classes imblance, since LR optimizes global loss, so it prefers dominant classes and neglecting the minority such as customer support, or HR.

However, since the label "Other" is ambiguous, and account for most of the dataset. We apply similar logic as before to categorize the this label. Those observations with top prediction lesser than 50% will be categoried into "Other" instead.

In [43]:
test_df = df_t_annotated_rb.copy()
X_test = test_df["position_norm"]
y_test = test_df["department"]

X_test_counts = count_vect.transform(X_test)
X_test_tfidf = tfidf_transformer.transform(X_test_counts)

y_proba = clf.predict_proba(X_test_tfidf)
y_pred = clf.classes_[np.argmax(y_proba, axis=1)]
y_pred_confidence_lv = np.max(y_proba, axis=1)

test_df["y_pred"] = y_pred
test_df["y_pred_confidence_lv"] = y_pred_confidence_lv

In [44]:
threshold = 0.5

test_df["y_pred_after_confidence_lv"] = np.where(
    test_df["y_pred_confidence_lv"] < threshold,
    "Other",
    test_df["y_pred"]
)

print("Accuracy with confidence threshold:", round(accuracy_score(test_df["department"], test_df["y_pred_after_confidence_lv"]) * 100, 2),"%")

print(classification_report(test_df["department"], test_df["y_pred_after_confidence_lv"]))

Accuracy with confidence threshold: 58.26 %
                        precision    recall  f1-score   support

        Administrative       0.14      0.04      0.06        84
  Business Development       0.45      0.38      0.41        78
            Consulting       0.88      0.43      0.57       195
      Customer Support       1.00      0.02      0.04        48
       Human Resources       1.00      0.30      0.46        70
Information Technology       1.00      0.04      0.09       312
             Marketing       0.68      0.20      0.31       133
                 Other       0.54      0.95      0.69      1252
    Project Management       0.98      0.35      0.51       175
            Purchasing       0.00      0.00      0.00        72
                 Sales       0.96      0.47      0.63       219

              accuracy                           0.58      2638
             macro avg       0.69      0.29      0.34      2638
          weighted avg       0.68      0.58      0.51     

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Since "Other" is the dominant class, the higher accuracy from the baseline model can be deceptive, as we replaced all "None" values with "Other" for our preidction. This approach, however, is better at generalizing to most cases, compared to the baseline model, even when its accuracy does not outperform. By leveraging confidence level, we can capture the ambiguous nature of "Other" labelling.

##Feature engineering and conventional machine learning

###Feature engineering

In [45]:
#feature engineering for train df

#test with different programmatic labels

#df = pd.read_csv('/content/df_t_notannotated_pred_embeddings_no_threshold.csv')
#df = pd.read_csv('/content/df_t_notannotated_pred_embeddings_both_threshold.csv')
#df = pd.read_csv('/content/df_t_notannotated_pred_embeddings_no_threshold.csv')
#df = pd.read_csv('/content/df_t_notannotated_pred_embeddings_both_threshold.csv')
df = pd.read_csv('/content/df_notannotated_pred_embedding_bow.csv')

df["position_norm"] = df["position"].map(normalize_text)
#df = df_t_notannotated_for_training.copy()


df["startDate"] = pd.to_datetime(df["startDate"], errors="coerce")
df["endDate"] = pd.to_datetime(df["endDate"], errors="coerce")
df["endDate_filled"] = df["endDate"].fillna(pd.Timestamp.today())

df["job_tenure_years"] = ((df["endDate_filled"] - df["startDate"]).dt.days / 365)
df["experience_until_job"] = (df.sort_values(["person_id", "startDate"]).groupby("person_id")["job_tenure_years"].cumsum())

df["has_sales_kw"] = df["position_norm"].str.contains("sales|account|business development",regex=True).astype(int)
df["has_it_kw"] = df["position_norm"].str.contains(r"software|developer|engineer|\b(it)\b|systems",regex=True,na=False).astype(int)
df["has_hr_kw"] = df["position_norm"].str.contains(r"\b(hr)\b|recruit|talent|people",regex=True).astype(int)
df["is_consulting_firm"] = df["organization"].str.contains("consult|deloitte|pwc|kpmg|ey",regex=True,na=False).astype(int)


career_feats = (df.groupby("person_id").agg(num_jobs=("job_no", "max"),total_experience=("job_tenure_years", "sum"),avg_tenure=("job_tenure_years", "mean")).reset_index())
df = df.merge(career_feats, on="person_id", how="left")

#df["title_len"] = df["position_norm"].str.len()
#df["has_manager"] = df["position_norm"].str.contains("manager", regex=False).astype(int)
#df["has_senior"] = df["position_norm"].str.contains("senior|lead|principal", regex=True).astype(int)

/tmp/ipython-input-1141291872.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["has_it_kw"] = df["position_norm"].str.contains(r"software|developer|engineer|\b(it)\b|systems",regex=True,na=False).astype(int)
/tmp/ipython-input-1141291872.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["has_hr_kw"] = df["position_norm"].str.contains(r"\b(hr)\b|recruit|talent|people",regex=True).astype(int)


In [46]:
dept_counts = (
    df
    .groupby("department")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

dept_counts

,department,count
7,Other,1095
2,Consulting,186
10,Sales,137
8,Project Management,93
1,Business Development,75
5,Information Technology,73
0,Administrative,71
6,Marketing,68
4,Human Resources,39
9,Purchasing,33


In [47]:
#feature engineering for test df
df_test = df_t_annotated_rb.copy()
df_test["startDate"] = pd.to_datetime(df_test["startDate"], errors="coerce")
df_test["endDate"] = pd.to_datetime(df_test["endDate"], errors="coerce")
df_test["endDate_filled"] = df_test["endDate"].fillna(pd.Timestamp.today())

df_test["job_tenure_years"] = ((df_test["endDate_filled"] - df_test["startDate"]).dt.days / 365)
df_test["experience_until_job"] = (df_test.sort_values(["person_id", "startDate"]).groupby("person_id")["job_tenure_years"].cumsum())

df_test["has_sales_kw"] = df_test["position_norm"].str.contains("sales|account|business development",regex=True).astype(int)
df_test["has_it_kw"] = df_test["position_norm"].str.contains(r"software|developer|engineer|\b(it)\b|systems",regex=True,na=False).astype(int)
df_test["has_hr_kw"] = df_test["position_norm"].str.contains(r"\b(hr)\b|recruit|talent|people",regex=True).astype(int)
df_test["is_consulting_firm"] = df_test["organization"].str.contains("consult|deloitte|pwc|kpmg|ey",regex=True,na=False).astype(int)

career_feats = (df_test.groupby("person_id").agg(num_jobs=("job_no", "max"),total_experience=("job_tenure_years", "sum"),avg_tenure=("job_tenure_years", "mean")).reset_index())
df_test = df_test.merge(career_feats, on="person_id", how="left")

/tmp/ipython-input-1416808472.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_test["has_it_kw"] = df_test["position_norm"].str.contains(r"software|developer|engineer|\b(it)\b|systems",regex=True,na=False).astype(int)
/tmp/ipython-input-1416808472.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_test["has_hr_kw"] = df_test["position_norm"].str.contains(r"\b(hr)\b|recruit|talent|people",regex=True).astype(int)


In [49]:
df

,person_id,job_no,organization,linkedin,position,startDate,endDate,status,department,seniority,...,endDate_filled,job_tenure_years,experience_until_job,has_sales_kw,has_it_kw,has_hr_kw,is_consulting_firm,num_jobs,total_experience,avg_tenure
0,1,1,Domino's Pizza,NaN,delivery/sales/MIT,2012-04-01,2012-06-01,INACTIVE,Sales,Professional,...,2012-06-01 00:00:00.000000,0.167123,0.167123,1,0,0,0,8,17.158904,2.144863
1,1,2,UPS Freight,https://www.linkedin.com/company/ups-freight,Regional driver,2016-10-01,2017-03-01,INACTIVE,Other,Professional,...,2017-03-01 00:00:00.000000,0.413699,0.580822,0,0,0,0,8,17.158904,2.144863
2,1,3,Sysco,https://www.linkedin.com/company/sysco,Food Delivery Driver,2017-10-01,2018-02-01,INACTIVE,Other,Professional,...,2018-02-01 00:00:00.000000,0.336986,0.917808,0,0,0,0,8,17.158904,2.144863
3,1,4,MB Railways,NaN,Fuel Driver,2018-03-01,2019-03-01,INACTIVE,Other,Professional,...,2019-03-01 00:00:00.000000,1.000000,1.917808,0,0,0,0,8,17.158904,2.144863
4,1,5,Playful Paws,NaN,Co-Owner,2018-11-01,NaT,ACTIVE,Other,Management,...,2026-01-26 15:51:51.234168,7.241096,9.158904,0,0,0,0,8,17.158904,2.144863
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1881,389,1,Euler Hermes Deutschland,https://www.linkedin.com/company/allianz-trade...,Juristischer Berater,1996-03-01,NaT,ACTIVE,Consulting,Professional,...,2026-01-26 15:51:51.234168,29.926027,29.926027,0,0,0,0,1,29.926027,29.926027
1882,390,1,Folkwang Universität der Künste,NaN,Dekanat,1998-02-01,2004-02-01,INACTIVE,Other,Professional,...,2004-02-01 00:00:00.000000,6.002740,6.002740,0,0,0,0,4,27.758904,6.939726
1883,390,2,Folkwang Universität der Künste,NaN,"Leitung Gebäudemanagement, Einkauf und Control...",2004-03-01,2005-09-01,INACTIVE,Purchasing,Lead,...,2005-09-01 00:00:00.000000,1.504110,7.506849,0,0,0,0,4,27.758904,6.939726
1884,390,3,Landesspracheninstitut in der Ruhr-Universität...,NaN,Verwaltungsleitung Landesspracheninstitut in d...,2005-10-01,2015-06-01,INACTIVE,Other,Lead,...,2015-06-01 00:00:00.000000,9.671233,17.178082,0,0,0,0,4,27.758904,6.939726


###Apply Logistics Regression

####Seniority

In [66]:
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

NUM_COLS = ["job_no","experience_until_job","job_tenure_years"]

df_train = df.dropna(subset=["seniority"]).copy()
X_text = df_train["position_norm"].astype(str)
X_num  = df_train[NUM_COLS].fillna(0)
y = df_train["seniority"]

#train-test split
X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_num, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#tf-idf + vectorize
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2
)

X_text_train_tfidf = tfidf.fit_transform(X_text_train)
X_text_test_tfidf  = tfidf.transform(X_text_test)

#Scale numeric features
scaler = StandardScaler()
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled  = scaler.transform(X_num_test)

# Convert numeric arrays to sparse so we can hstack with TF-IDF (sparse)
X_num_train_sparse = csr_matrix(X_num_train_scaled)
X_num_test_sparse  = csr_matrix(X_num_test_scaled)

#Combine TF-IDF and numeric features
X_train_final = hstack([X_text_train_tfidf, X_num_train_sparse])
X_test_final  = hstack([X_text_test_tfidf,  X_num_test_sparse])


#train LR and evaluate on test set
clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)  #balanced to punish underepresented observations when predict incorrectly

clf.fit(X_train_final, y_train)
y_pred = clf.predict(X_test_final)
print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))

Accuracy: 73.81 %
              precision    recall  f1-score   support

    Director       0.96      0.96      0.96        27
      Junior       0.64      0.68      0.66        40
        Lead       0.78      0.70      0.74        44
  Management       0.83      0.76      0.80        59
Professional       0.74      0.65      0.69       160
      Senior       0.61      0.96      0.75        48

    accuracy                           0.74       378
   macro avg       0.76      0.79      0.77       378
weighted avg       0.75      0.74      0.74       378



In [67]:
# new fe for dep

X_text_full = df_train["position_norm"].astype(str)
X_num_full  = df_train[NUM_COLS].fillna(0)
# Text → TF-IDF (transform only!)
X_text_full_tfidf = tfidf.transform(X_text_full)

# Numeric → scaled (transform only!)
X_num_full_scaled = scaler.transform(X_num_full)

# Convert numeric to sparse
X_num_full_sparse = csr_matrix(X_num_full_scaled)

X_full_final = hstack([
    X_text_full_tfidf,
    X_num_full_sparse
])

y_pred_full = clf.predict(X_full_final)

y_proba_full = clf.predict_proba(X_full_final)
y_conf_full  = y_proba_full.max(axis=1)

df_train_fe_for_dep = df_train.copy()

df_train_fe_for_dep["pred_seniority"] = y_pred_full
df_train_fe_for_dep["pred_seniority_conf"] = y_conf_full

seniority_order = {
    "Junior": 1,
    "Professional": 2,
    "Senior": 3,
    "Lead": 4,
    "Management": 5,
    "Director": 6
}

df_train_fe_for_dep = df_train_fe_for_dep.sort_values(["person_id", "startDate", "job_no"]).copy()
df_train_fe_for_dep["prev_department"] = df_train_fe_for_dep.groupby("person_id")["department"].shift(1)
prev_dept_dummies = pd.get_dummies(df_train_fe_for_dep["prev_department"],prefix="prev")
df_train_fe_for_dep["is_first_job"] = df_train_fe_for_dep["prev_department"].isna().astype(int)
df_train_fe_for_dep = pd.concat([df_train_fe_for_dep, prev_dept_dummies], axis=1)

df_train_fe_for_dep["pred_seniority_level"] = df_train_fe_for_dep["pred_seniority"].map(seniority_order)
df_train_fe_for_dep["startDate"] = pd.to_datetime(df_train_fe_for_dep["startDate"], errors="coerce")

df_train_fe_for_dep = df_train_fe_for_dep.sort_values(["person_id", "startDate", "job_no"])
df_train_fe_for_dep["prev_pred_seniority"] = df_train_fe_for_dep.groupby("person_id")["pred_seniority"].shift(1)
df_train_fe_for_dep["prev_pred_seniority_level"] = df_train_fe_for_dep.groupby("person_id")["pred_seniority_level"].shift(1)
df_train_fe_for_dep["seniority_change"] = (
    df_train_fe_for_dep["pred_seniority_level"] - df_train_fe_for_dep["prev_pred_seniority_level"]
)
df_train_fe_for_dep["is_promotion"] = (df_train_fe_for_dep["seniority_change"] > 0).astype(int)
df_train_fe_for_dep["is_lateral"]   = (df_train_fe_for_dep["seniority_change"] == 0).astype(int)
df_train_fe_for_dep["is_demotion"]  = (df_train_fe_for_dep["seniority_change"] < 0).astype(int)
df_train_fe_for_dep["is_first_job"] = df_train_fe_for_dep["prev_pred_seniority_level"].isna().astype(int)
df_train_fe_for_dep["likely_dept_change"] = df_train_fe_for_dep["is_demotion"]
df_train_fe_for_dep["cum_promotions"] = df_train_fe_for_dep.groupby("person_id")["is_promotion"].cumsum()
df_train_fe_for_dep["cum_demotions"]  = df_train_fe_for_dep.groupby("person_id")["is_demotion"].cumsum()
df_train_fe_for_dep["max_seniority_so_far"] = df_train_fe_for_dep.groupby("person_id")["pred_seniority_level"].cummax()
df_train_fe_for_dep["max_seniority_so_far"] = df_train_fe_for_dep.groupby("person_id")["pred_seniority_level"].cummax()
df_train_fe_for_dep["gap_to_best_so_far"] = df_train_fe_for_dep["max_seniority_so_far"] - df_train_fe_for_dep["pred_seniority_level"]
cols_check = [
    "person_id", "job_no", "startDate", "position_norm",
    "pred_seniority", "prev_pred_seniority",
    "seniority_change", "is_promotion", "is_lateral", "is_demotion",
    "cum_promotions", "cum_demotions", "max_seniority_so_far", "gap_to_best_so_far",
    "likely_dept_change"
]

#df_train_fe_for_dep[cols_check].head(10)
#df_train_fe_for_dep.info()

In [68]:
# Performance on annotated dataset

#eval_df = (df_test[df_test["seniority"] != "Professional"].copy())
#eval_df = df_test[df_test["status"] == "ACTIVE"].copy()
eval_df = df_test.copy()

X_eval_text = eval_df["position_norm"].astype(str)
X_eval_num  = eval_df[NUM_COLS].fillna(0)

X_eval_text_tfidf = tfidf.transform(X_eval_text)
X_eval_num_scaled = scaler.transform(X_eval_num)
X_eval_final = hstack([X_eval_text_tfidf, csr_matrix(X_eval_num_scaled)])

y_eval_pred = clf.predict(X_eval_final)
y_eval_proba = clf.predict_proba(X_eval_final)
eval_conf = y_eval_proba.max(axis=1)

y_eval_pred_conf = np.where(eval_conf < 0.0, "Professional", y_eval_pred)

print("Eval accuracy:", round(accuracy_score(eval_df["seniority"], y_eval_pred_conf) * 100, 2), "%")
print(classification_report(eval_df["seniority"], y_eval_pred_conf))

Eval accuracy: 57.05 %
              precision    recall  f1-score   support

    Director       0.71      0.86      0.78       141
      Junior       0.51      0.64      0.57       230
        Lead       0.65      0.54      0.59       460
  Management       0.74      0.65      0.69       418
Professional       0.64      0.53      0.58      1219
      Senior       0.18      0.46      0.26       170

    accuracy                           0.57      2638
   macro avg       0.57      0.61      0.58      2638
weighted avg       0.62      0.57      0.59      2638



In [69]:
#to save df for later feature engineering (annotated file)
eval_df = eval_df.copy()
eval_df["pred_seniority"] = y_eval_pred_conf
df_test.loc[eval_df.index, "pred_seniority"] = y_eval_pred_conf

df_test_fe_for_dep = df_test.copy()

df_test_fe_for_dep = df_test.sort_values(["person_id", "startDate", "job_no"]).copy()
df_test_fe_for_dep["prev_department"] = df_test_fe_for_dep.groupby("person_id")["department"].shift(1)
prev_dept_dummies = pd.get_dummies(df_test_fe_for_dep["prev_department"],prefix="prev")
df_test_fe_for_dep["is_first_job"] = df_test_fe_for_dep["prev_department"].isna().astype(int)
df_test_fe_for_dep = pd.concat([df_test_fe_for_dep, prev_dept_dummies], axis=1)

df_test_fe_for_dep["pred_seniority_level"] = df_test_fe_for_dep["pred_seniority"].map(seniority_order)
df_test_fe_for_dep["startDate"] = pd.to_datetime(df_test_fe_for_dep["startDate"], errors="coerce")

df_test_fe_for_dep = df_test_fe_for_dep.sort_values(["person_id", "startDate", "job_no"])
df_test_fe_for_dep["prev_pred_seniority"] = df_test_fe_for_dep.groupby("person_id")["pred_seniority"].shift(1)
df_test_fe_for_dep["prev_pred_seniority_level"] = df_test_fe_for_dep.groupby("person_id")["pred_seniority_level"].shift(1)
df_test_fe_for_dep["seniority_change"] = (
    df_test_fe_for_dep["pred_seniority_level"] - df_test_fe_for_dep["prev_pred_seniority_level"]
)
df_test_fe_for_dep["is_promotion"] = (df_test_fe_for_dep["seniority_change"] > 0).astype(int)
df_test_fe_for_dep["is_lateral"]   = (df_test_fe_for_dep["seniority_change"] == 0).astype(int)
df_test_fe_for_dep["is_demotion"]  = (df_test_fe_for_dep["seniority_change"] < 0).astype(int)
df_test_fe_for_dep["is_first_job"] = df_test_fe_for_dep["prev_pred_seniority_level"].isna().astype(int)
df_test_fe_for_dep["likely_dept_change"] = df_test_fe_for_dep["is_demotion"]
df_test_fe_for_dep["cum_promotions"] = df_test_fe_for_dep.groupby("person_id")["is_promotion"].cumsum()
df_test_fe_for_dep["cum_demotions"]  = df_test_fe_for_dep.groupby("person_id")["is_demotion"].cumsum()
df_test_fe_for_dep["max_seniority_so_far"] = df_test_fe_for_dep.groupby("person_id")["pred_seniority_level"].cummax()
df_test_fe_for_dep["max_seniority_so_far"] = df_test_fe_for_dep.groupby("person_id")["pred_seniority_level"].cummax()
df_test_fe_for_dep["gap_to_best_so_far"] = df_test_fe_for_dep["max_seniority_so_far"] - df_test_fe_for_dep["pred_seniority_level"]
cols_check = [
    "person_id", "job_no", "startDate", "position_norm",
    "pred_seniority", "prev_pred_seniority",
    "seniority_change", "is_promotion", "is_lateral", "is_demotion",
    "cum_promotions", "cum_demotions", "max_seniority_so_far", "gap_to_best_so_far",
    "likely_dept_change"
]

#df_test_fe_for_dep[cols_check].head(30)
pd.set_option('display.max_columns', 100)
#df_test_fe_for_dep.head(30)

In [70]:
# @title
from sklearn.metrics import confusion_matrix

labels = sorted(eval_df["seniority"].unique())

cm = confusion_matrix(
    eval_df["seniority"],
    y_eval_pred_conf,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{l}" for l in labels],
    columns=[f"pred_{l}" for l in labels]
)

cm_df

,pred_Director,pred_Junior,pred_Lead,pred_Management,pred_Professional,pred_Senior
true_Director,121,0,3,3,14,0
true_Junior,1,148,7,4,64,6
true_Lead,1,8,247,14,155,35
true_Management,44,0,22,270,61,21
true_Professional,2,122,99,65,640,291
true_Senior,2,15,3,8,63,79


####Department

In [55]:
#NUM_COLS = ["job_no"]
NUM_COLS = ["has_sales_kw","has_it_kw","has_hr_kw","is_consulting_firm"]
#NUM_COLS = ["job_no","experience_until_job","job_tenure_years"]
#NUM_COLS = ['gap_to_best_so_far', 'likely_dept_change', "max_seniority_so_far"]


df_train = df.dropna(subset=["department"]).copy()
X_text = df_train["position_norm"].astype(str)
X_num  = df_train[NUM_COLS].fillna(0)
y = df_train["department"]

#train-test split
X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_num, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#tf-idf + vectorize
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2
)

X_text_train_tfidf = tfidf.fit_transform(X_text_train)
X_text_test_tfidf  = tfidf.transform(X_text_test)

#Scale numeric features
scaler = StandardScaler()
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled  = scaler.transform(X_num_test)

# Convert numeric arrays to sparse so we can hstack with TF-IDF (sparse)
X_num_train_sparse = csr_matrix(X_num_train_scaled)
X_num_test_sparse  = csr_matrix(X_num_test_scaled)

#Combine TF-IDF and numeric features
X_train_final = hstack([X_text_train_tfidf, X_num_train_sparse])
X_test_final  = hstack([X_text_test_tfidf,  X_num_test_sparse])


#train LR and evaluate on test set
clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)  #balanced to punish underepresented observations when predict incorrectly

clf.fit(X_train_final, y_train)
y_pred = clf.predict(X_test_final)
print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))

Accuracy: 79.1 %
                        precision    recall  f1-score   support

        Administrative       0.65      0.79      0.71        14
  Business Development       0.69      0.73      0.71        15
            Consulting       0.75      0.81      0.78        37
      Customer Support       1.00      0.67      0.80         3
       Human Resources       0.64      0.88      0.74         8
Information Technology       0.30      0.53      0.38        15
             Marketing       0.93      0.93      0.93        14
                 Other       0.85      0.81      0.83       219
    Project Management       1.00      0.84      0.91        19
            Purchasing       0.71      0.71      0.71         7
                 Sales       0.95      0.67      0.78        27

              accuracy                           0.79       378
             macro avg       0.77      0.76      0.75       378
          weighted avg       0.82      0.79      0.80       378



In [56]:
# Performance on annotated dataset

#eval_df = (df_test[df_test["department"] != "Other"].copy())
#eval_df = df_test[df_test["status"] == "ACTIVE"].copy()
eval_df = df_test.copy()

X_eval_text = eval_df["position_norm"].astype(str)
X_eval_num  = eval_df[NUM_COLS].fillna(0)

X_eval_text_tfidf = tfidf.transform(X_eval_text)
X_eval_num_scaled = scaler.transform(X_eval_num)
X_eval_final = hstack([X_eval_text_tfidf, csr_matrix(X_eval_num_scaled)])

y_eval_pred = clf.predict(X_eval_final)
y_eval_proba = clf.predict_proba(X_eval_final)
eval_conf = y_eval_proba.max(axis=1)

y_eval_pred_conf = np.where(eval_conf < 0.5, "Other", y_eval_pred)  # ignore this confidence Other replacement with department

print("Eval accuracy:", round(accuracy_score(eval_df["department"], y_eval_pred_conf) * 100, 2), "%")
print(classification_report(eval_df["department"], y_eval_pred_conf))

Eval accuracy: 64.82 %
                        precision    recall  f1-score   support

        Administrative       0.19      0.12      0.15        84
  Business Development       0.39      0.45      0.42        78
            Consulting       0.78      0.49      0.60       195
      Customer Support       0.77      0.35      0.49        48
       Human Resources       0.90      0.61      0.73        70
Information Technology       0.89      0.30      0.45       312
             Marketing       0.68      0.27      0.39       133
                 Other       0.61      0.91      0.73      1252
    Project Management       0.99      0.45      0.61       175
            Purchasing       0.90      0.26      0.41        72
                 Sales       0.88      0.63      0.74       219

              accuracy                           0.65      2638
             macro avg       0.73      0.44      0.52      2638
          weighted avg       0.70      0.65      0.62      2638



In [57]:
# @title
labels = sorted(eval_df["department"].unique())

cm = confusion_matrix(
    eval_df["department"],
    y_eval_pred_conf,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{l}" for l in labels],
    columns=[f"pred_{l}" for l in labels]
)

cm_df

,pred_Administrative,pred_Business Development,pred_Consulting,pred_Customer Support,pred_Human Resources,pred_Information Technology,pred_Marketing,pred_Other,pred_Project Management,pred_Purchasing,pred_Sales
true_Administrative,10,2,0,0,0,0,0,72,0,0,0
true_Business Development,0,35,12,0,0,0,0,30,0,0,1
true_Consulting,1,12,96,0,3,1,2,80,0,0,0
true_Customer Support,1,0,0,17,0,0,0,28,0,0,2
true_Human Resources,0,0,1,0,43,0,0,25,0,0,1
true_Information Technology,1,5,0,5,0,93,0,208,0,0,0
true_Marketing,2,1,1,0,0,1,36,91,0,1,0
true_Other,34,30,13,0,2,3,12,1145,1,0,12
true_Project Management,1,3,0,0,0,5,0,86,78,0,2
true_Purchasing,0,0,0,0,0,0,0,53,0,19,0


####Feature engineering of deapartment extension

Job titles alone are often ambiguous and insufficient to accurately infer a candidate’s department (e.g., “Manager”, “Consultant”). However, career paths are sequential and path-dependent: a person’s current department is strongly influenced by their previous department(s).

The goal of this feature engineering step is therefore to augment text-based classification with contextual career history, using previous prediction.

In [83]:
#NUM_COLS = ["job_no"]
#NUM_COLS = ["has_sales_kw","has_it_kw","has_hr_kw","is_consulting_firm"]
#NUM_COLS = ["job_no","experience_until_job","job_tenure_years"]
#NUM_COLS = ["seniority_change"]
NUM_COLS = ["has_sales_kw","has_it_kw","has_hr_kw","is_consulting_firm","prev_Administrative", "prev_Business Development", "prev_Consulting","prev_Customer Support","prev_Human Resources","prev_Information Technology","prev_Marketing","prev_Other","prev_Project Management","prev_Purchasing","prev_Sales"]


df_train = df_train_fe_for_dep.dropna(subset=["department"]).copy()
X_text = df_train["position_norm"].astype(str)
X_num  = df_train[NUM_COLS].fillna(0)
y = df_train["department"]

#train-test split
X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_num, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#tf-idf + vectorize
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2
)

X_text_train_tfidf = tfidf.fit_transform(X_text_train)
X_text_test_tfidf  = tfidf.transform(X_text_test)

#Scale numeric features
scaler = StandardScaler()
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled  = scaler.transform(X_num_test)

# Convert numeric arrays to sparse so we can hstack with TF-IDF (sparse)
X_num_train_sparse = csr_matrix(X_num_train_scaled)
X_num_test_sparse  = csr_matrix(X_num_test_scaled)

#Combine TF-IDF and numeric features
X_train_final = hstack([X_text_train_tfidf, X_num_train_sparse])
X_test_final  = hstack([X_text_test_tfidf,  X_num_test_sparse])


#train LR and evaluate on test set
clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)  #balanced to punish underepresented observations when predict incorrectly

clf.fit(X_train_final, y_train)
y_pred = clf.predict(X_test_final)
print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))

Accuracy: 74.34 %
                        precision    recall  f1-score   support

        Administrative       0.52      0.79      0.63        14
  Business Development       0.71      0.67      0.69        15
            Consulting       0.72      0.78      0.75        37
      Customer Support       0.67      0.67      0.67         3
       Human Resources       0.64      0.88      0.74         8
Information Technology       0.29      0.60      0.39        15
             Marketing       0.85      0.79      0.81        14
                 Other       0.88      0.74      0.80       219
    Project Management       0.67      0.84      0.74        19
            Purchasing       0.56      0.71      0.62         7
                 Sales       0.70      0.70      0.70        27

              accuracy                           0.74       378
             macro avg       0.65      0.74      0.69       378
          weighted avg       0.78      0.74      0.75       378



In [84]:
# Performance on annotated dataset

#eval_df = (df_test[df_test["department"] != "Other"].copy())
#eval_df = df_test[df_test["status"] == "ACTIVE"].copy()
eval_df = df_test_fe_for_dep.copy()

X_eval_text = eval_df["position_norm"].astype(str)
X_eval_num  = eval_df[NUM_COLS].fillna(0)

X_eval_text_tfidf = tfidf.transform(X_eval_text)
X_eval_num_scaled = scaler.transform(X_eval_num)
X_eval_final = hstack([X_eval_text_tfidf, csr_matrix(X_eval_num_scaled)])

y_eval_pred = clf.predict(X_eval_final)
y_eval_proba = clf.predict_proba(X_eval_final)
eval_conf = y_eval_proba.max(axis=1)

y_eval_pred_conf = np.where(eval_conf < 0.5, "Other", y_eval_pred)  # ignore this confidence Other replacement with department

print("Eval accuracy:", round(accuracy_score(eval_df["department"], y_eval_pred_conf) * 100, 2), "%")
print(classification_report(eval_df["department"], y_eval_pred_conf))

Eval accuracy: 67.63 %
                        precision    recall  f1-score   support

        Administrative       0.25      0.12      0.16        84
  Business Development       0.40      0.45      0.42        78
            Consulting       0.83      0.46      0.59       195
      Customer Support       0.81      0.35      0.49        48
       Human Resources       0.90      0.61      0.73        70
Information Technology       0.88      0.50      0.63       312
             Marketing       0.65      0.30      0.41       133
                 Other       0.63      0.91      0.75      1252
    Project Management       0.97      0.43      0.60       175
            Purchasing       0.64      0.54      0.59        72
                 Sales       0.89      0.62      0.73       219

              accuracy                           0.68      2638
             macro avg       0.71      0.48      0.55      2638
          weighted avg       0.71      0.68      0.66      2638



In [85]:
PREV_COLS = [
    "prev_Administrative",
    "prev_Business Development",
    "prev_Consulting",
    "prev_Customer Support",
    "prev_Human Resources",
    "prev_Information Technology",
    "prev_Marketing",
    "prev_Other",
    "prev_Project Management",
    "prev_Purchasing",
    "prev_Sales",
]

NUM_COLS = ["has_sales_kw","has_it_kw","has_hr_kw","is_consulting_firm"]
NUM_FEATURES_USED = NUM_COLS + PREV_COLS

for c in NUM_FEATURES_USED:
    if c not in df_test_fe_for_dep.columns:
        df_test_fe_for_dep[c] = 0


import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import accuracy_score, classification_report

def predict_one_row(row_df, tfidf, scaler, clf, num_features_used):
    X_text = row_df["position_norm"].astype(str)
    X_num  = row_df[num_features_used].fillna(0)

    X_text_tfidf = tfidf.transform(X_text)
    X_num_scaled = scaler.transform(X_num)

    X_final = hstack([X_text_tfidf, csr_matrix(X_num_scaled)])
    return clf.predict(X_final)[0]


df_seq = df_test_fe_for_dep.copy()
df_seq["startDate"] = pd.to_datetime(df_seq["startDate"], errors="coerce")
df_seq = df_seq.sort_values(["person_id", "startDate", "job_no"]).copy()

# update previous prediction department
df_seq["prev_department"] = np.nan
df_seq["pred_department_seq"] = None

# ensure prev_ cols start at 0
df_seq[PREV_COLS] = 0

# sequential roll-forward per person
for pid, idxs in df_seq.groupby("person_id").groups.items():
    prev_pred = np.nan

    # idxs are already in sorted order because df_seq is sorted
    for idx in list(idxs):
        # set prev_department = previous prediction
        df_seq.loc[idx, "prev_department"] = prev_pred

        # reset prev_ one-hot
        df_seq.loc[idx, PREV_COLS] = 0

        # set the right prev_ column if we have a previous prediction
        if pd.notna(prev_pred):
            col = f"prev_{prev_pred}"
            if col in PREV_COLS:
                df_seq.loc[idx, col] = 1

        # predict current row
        pred = predict_one_row(df_seq.loc[[idx]], tfidf, scaler, clf, NUM_FEATURES_USED)

        df_seq.loc[idx, "pred_department_seq"] = pred
        prev_pred = pred  # roll forward


print("Accuracy:",
      round(accuracy_score(df_seq["department"], df_seq["pred_department_seq"]) * 100, 2), "%")
print(classification_report(df_seq["department"], df_seq["pred_department_seq"], zero_division=0))

/tmp/ipython-input-1005442224.py:57: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Other' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_seq.loc[idx, "prev_department"] = prev_pred


Accuracy: 56.52 %
                        precision    recall  f1-score   support

        Administrative       0.14      0.20      0.17        84
  Business Development       0.35      0.50      0.41        78
            Consulting       0.46      0.61      0.52       195
      Customer Support       0.57      0.44      0.49        48
       Human Resources       0.84      0.54      0.66        70
Information Technology       0.50      0.54      0.52       312
             Marketing       0.41      0.35      0.38       133
                 Other       0.69      0.62      0.65      1252
    Project Management       0.55      0.54      0.55       175
            Purchasing       0.25      0.36      0.30        72
                 Sales       0.69      0.68      0.68       219

              accuracy                           0.57      2638
             macro avg       0.50      0.49      0.48      2638
          weighted avg       0.59      0.57      0.57      2638

